In [ ]:
# manipulated
import gdown

url = "https://drive.google.com/uc?id=1LgWy39Pb-UUAZ4HpaUTADW0qqpsg8-hi"
gdown.download(url,'/content/manipulated.7z',quiet=False)

In [ ]:
import py7zr

archive_path = '/content/manipulated.7z'
output_dir = '/content'

with py7zr.SevenZipFile(archive_path, mode='r') as z:
    z.extractall(path=output_dir)

In [ ]:
# original

url = "https://drive.google.com/uc?id=17EFN2RcdaNI4HgkwWp2-KQ6Z7OzJUZlO"
gdown.download(url,'/content/original.zip',quiet=False)

In [ ]:
!unzip /content/original.zip -d /content/original

In [ ]:
import os
import glob

root_dir = '/content'  # 본인 폴더로 수정

for subdir in ['original', 'manipulated']:
    sub_path = os.path.join(root_dir, subdir)
    count = len(glob.glob(os.path.join(sub_path, '*.jpg')))
    print(f"{subdir} 폴더의 jpg 파일 개수: {count}개")


In [ ]:
folder = '/content/manipulated'  # 폴더 경로 수정

deleted_count = 0
for filepath in glob.glob(os.path.join(folder, '*.jpg')):
    filename = os.path.basename(filepath)
    # _0.jpg로 끝나지 않는 경우만 삭제
    if not filename.endswith('_0.jpg'):
        os.remove(filepath)
        deleted_count += 1

print(f'삭제한 jpg 파일 개수: {deleted_count}')

In [ ]:
folder = '/content/original'  # 폴더 경로 수정

deleted_count = 0
for filepath in glob.glob(os.path.join(folder, '*.jpg')):
    filename = os.path.basename(filepath)
    # _0.jpg로 끝나지 않는 경우만 삭제
    if not filename.endswith('_0.jpg'):
        os.remove(filepath)
        deleted_count += 1

print(f'삭제한 jpg 파일 개수: {deleted_count}')

In [ ]:
root_dir = '/content'

for subdir in ['original', 'manipulated']:
    sub_path = os.path.join(root_dir, subdir)
    count = len(glob.glob(os.path.join(sub_path, '*.jpg')))
    print(f"{subdir} 폴더의 jpg 파일 개수: {count}개")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import albumentations as A
import cv2
from tqdm import tqdm


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        BCE = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE)
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE
        return F_loss.mean()


In [ ]:
class DeepfakeVideoDataset(Dataset):
    def __init__(self, root_dir, frame_num=8, size=224, augment=False):
        self.data = []
        self.labels = []
        self.frame_num = frame_num
        self.size = size
        self.augment = augment
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        for label, subdir in enumerate(['original', 'manipulated']):
            sub_path = os.path.join(root_dir, subdir)
            video_dict = {}
            for img in glob.glob(os.path.join(sub_path, "*.jpg")):
                video_id = os.path.basename(img).split('_f')[0]
                if video_id not in video_dict:
                    video_dict[video_id] = []
                video_dict[video_id].append(img)
            for video_id, frame_list in video_dict.items():
                if len(frame_list) == self.frame_num:
                    frame_list.sort()  # f00~f07
                    self.data.append(frame_list)
                    self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        frames = []
        for img_path in self.data[idx]:
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.size, self.size))
            frames.append(self.transform(img))
        frames = torch.stack(frames, dim=1)  # (C, T, H, W) for swin3d
        label = self.labels[idx]
        video_id = os.path.basename(self.data[idx][0]).split('_f')[0]
        return frames, label, video_id


In [ ]:
# 데이터셋 경로
root_dir = '/content'

dataset = DeepfakeVideoDataset(root_dir, frame_num=8, size=224)

# train:val = 8:2 (stratified X, shuffle O)
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=4, shuffle=False, num_workers=2)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 모델 가중치 불러오기 및 이진 분류를 위해 헤드 변경, gpu 설정
weights = Swin3D_T_Weights.DEFAULT
model = swin3d_t(weights=weights)
model.head = nn.Linear(model.head.in_features, 2)  # 이진 분류
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = FocalLoss(alpha=0.25, gamma=2)  # 포컬로스 클래스 비율 맞게 alpha 조절하기


In [ ]:
best_auc = 0
patience = 10
pat = 0
num_epochs = 30

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'train_f1':[], 'val_f1':[], 'train_auc':[], 'val_auc':[]}

for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    train_losses, preds, targets = [], [], []
    for x, y, vid in tqdm(train_loader, desc=f'Epoch {epoch+1} [Train]'):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        preds += out.softmax(1)[:,1].detach().cpu().tolist()
        targets += y.cpu().tolist()
    train_loss = np.mean(train_losses)
    train_pred_label = [1 if p > 0.5 else 0 for p in preds]
    train_acc = accuracy_score(targets, train_pred_label)
    train_f1 = f1_score(targets, train_pred_label)
    train_auc = roc_auc_score(targets, preds)

    # --- Validation ---
    model.eval()
    val_losses, preds, targets = [], [], []
    with torch.no_grad():
        for x, y, vid in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]'):
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            val_losses.append(loss.item())
            preds += out.softmax(1)[:,1].cpu().tolist()
            targets += y.cpu().tolist()
    val_loss = np.mean(val_losses)
    val_pred_label = [1 if p > 0.5 else 0 for p in preds]
    val_acc = accuracy_score(targets, val_pred_label)
    val_f1 = f1_score(targets, val_pred_label)
    val_auc = roc_auc_score(targets, preds)

    # 기록
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)

    print(f"[{epoch+1}] train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}, val_auc={val_auc:.4f}")

    # Early Stopping & Checkpoint
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'best_swin3d_tiny.pth')
        pat = 0
    else:
        pat += 1
    if pat >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break


In [ ]:
epochs = len(history['train_loss'])
plt.figure(figsize=(12,8))
plt.subplot(2,2,1)
plt.plot(range(epochs), history['train_loss'], label='Train')
plt.plot(range(epochs), history['val_loss'], label='Val')
plt.title('Loss')
plt.grid()
plt.legend()

plt.subplot(2,2,2)
plt.plot(range(epochs), history['train_acc'], label='Train')
plt.plot(range(epochs), history['val_acc'], label='Val')
plt.title('Accuracy')
plt.grid()
plt.legend()

plt.subplot(2,2,3)
plt.plot(range(epochs), history['train_f1'], label='Train')
plt.plot(range(epochs), history['val_f1'], label='Val')
plt.title('F1-score')
plt.grid()
plt.legend()

plt.subplot(2,2,4)
plt.plot(range(epochs), history['train_auc'], label='Train')
plt.plot(range(epochs), history['val_auc'], label='Val')
plt.title('AUC')
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.2)

epochs = len(history['train_loss'])
x = np.arange(1, epochs + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
titles = ['Loss', 'Accuracy', 'F1-score', 'AUC']
metrics = [
    ('train_loss', 'val_loss'),
    ('train_acc', 'val_acc'),
    ('train_f1', 'val_f1'),
    ('train_auc', 'val_auc')
]

train_color = '#0066CC'  # 진한 파랑
val_color = '#FF5733'    # 오렌지/빨강

for idx, ax in enumerate(axes.flat):
    tr, va = metrics[idx]
    ax.plot(x, history[tr], label='Train', color=train_color, linewidth=2.2, marker='o', markersize=6)
    ax.plot(x, history[va], label='Val', color=val_color, linewidth=2.2, marker='s', markersize=6)
    ax.set_title(titles[idx], fontsize=17, weight='bold')
    ax.set_xlabel('Epoch', fontsize=13)
    ax.set_ylabel(titles[idx], fontsize=13)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=12, loc='best')
    # Best epoch marker for val
    best_idx = np.argmax(history[va]) if 'loss' not in va else np.argmin(history[va])
    ax.scatter(x[best_idx], history[va][best_idx],
               color=val_color, s=120, edgecolor='black',
               label='Best Val', zorder=5)
    if idx == 0:  # Loss만 ymin 조정
        ax.set_ylim(bottom=0)

plt.suptitle("Training vs. Validation Metrics per Epoch", fontsize=22, weight='bold', color='#1a1a1a', y=1.03)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# validation 데이터로 예측 (best model 기준)
model.load_state_dict(torch.load('best_swin3d_tiny.pth'))
model.eval()

preds, targets, video_ids = [], [], []
with torch.no_grad():
    for x, y, vid in tqdm(val_loader):
        x = x.to(device)
        out = model(x)
        probs = out.softmax(1)[:, 1].cpu().tolist()  # manipulated 확률
        preds += probs
        targets += y.cpu().tolist()
        video_ids += list(vid)  # video_id를 리스트에 누적

val_pred_label = [1 if p > 0.5 else 0 for p in preds]
cm = confusion_matrix(targets, val_pred_label)
disp = ConfusionMatrixDisplay(cm, display_labels=['Original', 'Manipulated'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(cmap='Blues', ax=ax)
ax.grid(False)   # grid(격자) 끄기
plt.show()

# 오분류된 샘플 뽑기
mis_idx = [i for i, (t, p) in enumerate(zip(targets, val_pred_label)) if t != p]
misclassified = [(video_ids[i], targets[i], val_pred_label[i], preds[i]) for i in mis_idx]

print(f"\n오분류된 샘플 수: {len(misclassified)}")
print("video_id, 실제정답, 예측값, 예측확률")
for v_id, label, pred, prob in misclassified[:20]:  # 앞 20개만 예시 출력
    print(f"{v_id:15s}  정답: {label}  예측: {pred}  (manip prob: {prob:.2f})")